### Loading in Packages

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py


### Loading in Data

In [14]:
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc',chunks={"time": 40}) #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***

In [13]:
# Loading Important Variables
##############
if 'emptylike' not in globals():
    print('loading neccessary variables')
    variable='w'; w_data=data[variable] #get w data
    w_data=w_data.interp(zf=data['zh']).data #interpolation w data z coordinate from zh to zf
    variable='qc'; qc_data=data[variable].data # get qc data
    variable='qi'; qi_data=data[variable].data # get qc data
    qc_plus_qi=qc_data+qi_data
    print('done')
    empty_like=True

loading neccessary variables
done


In [32]:
t=30;z=5;y=50;x=256
# w_data[t,z,y,x].compute()

-0.12237455

In [35]:
qc_data

dask.array<open_dataset-qc, shape=(141, 34, 100, 512), dtype=float32, chunksize=(40, 34, 100, 512), chunktype=numpy.ndarray>

In [30]:
#Lagrangian Position Arrays
##############
def grid_location(x,y,z): #faster
    #finding xf and yf
    ybins=data['yf'].values*1000; dy=ybins[1]-ybins[0] #1000
    xbins=data['xf'].values*1000; dx=xbins[1]-xbins[0] #1000
    dy=np.round(dy);dx=np.round(dx)

    #digitizing
    zf=data['zf'].values*1000; which_zh=np.searchsorted(zf,z)-1; which_zh=np.where(which_zh == -1, 0, which_zh) #finds which z layer parcel in 
    if which_zh.ndim==0:
        which_zh=np.array([which_zh])
    which_yh=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0]
    which_xh=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0]

    #fixing boundaries
    which_zh[np.where(which_zh==len(data['zh']))]-=1
    which_yh[np.where(which_yh==len(data['yh']))]-=1
    which_xh[np.where(which_xh==len(data['xh']))]-=1
    return which_zh,which_yh.values,which_xh.values
# x=parcel['x'].data;y=parcel['y'].data;z=parcel['z'].data
x=parcel['x'];y=parcel['y'];z=parcel['z']
Z,Y,X=grid_location(x,y,z)

Z

array([[30, 19, 28, ..., 33, 23, 24],
       [30, 19, 28, ..., 33, 23, 24],
       [29, 19, 28, ..., 33, 23, 24],
       ...,
       [30, 18, 27, ..., 33, 22, 23],
       [30, 18, 27, ..., 33, 22, 23],
       [29, 18, 27, ..., 33, 22, 23]])

In [7]:
W=np.zeros_like(Z,dtype='float32')
QCQI=np.zeros_like(Z,dtype='float32')

Nt=len(data['time'])
Np=len(parcel['xh'])
for p in np.arange(Np):
    if np.mod(p,25e3)==0: print(f"{p}/{len(parcel['xh'])}")

    #Get Indices
    ts = np.arange(Nt)  
    zs=Z[:,p]
    ys=Y[:,p]
    xs=X[:,p]
    
    #Get Values
    ws = w_data[ts, zs, ys, xs] # all data is already into memory. 
    
    qcqi = qc_plus_qi[ts, zs, ys, xs]

    #Store Values
    W[:,p]=ws
    QCQI[:,p]=qcqi
    #TESTING
    # for i in np.arange(w_data.shape[0]):
    #     test=(w_data[i,zs[i],ys[i],xs[i]]==hey2[i])
    #     if test==False:
    #         print('false')

0/125000
25000/125000
50000/125000
75000/125000
100000/125000


In [14]:
#Create Set Thresholds and Create Binary Arrays
w_thresh1=0.1
w_thresh2=0.5
qcqi_thresh=1e-6

# shape
t,z,y,x

#Apply Thresholds 
A_g = ( (W >= w_thresh1) & (QCQI < qcqi_thresh) )
A_c = ( (W >= w_thresh2) & (QCQI >= qcqi_thresh) )

# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'lagrangian_binary_array.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('A_g', data=A_g) #binary array for general updraft (w>=0.1 & qc+qi<1e-6)
    f.create_dataset('A_c', data=A_c) #binary array for general updraft (w>=0.5 & qc+qi>=1e-6)

    f.create_dataset('W', data=W)
    f.create_dataset('QCQI', data=QCQI)
    f.create_dataset('Z', data=Z)
    f.create_dataset('Y', data=Y)
    f.create_dataset('X', data=X)

In [ ]:
#########################################

In [15]:
# # Reading Back Data Later
# ##############
# import h5py
# dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# with h5py.File(dir2+'lagrangian_binary_array.h5', 'r') as f:
#     # Load the dataset by its name
#     A_g = f['A_g'][:]
#     A_c = f['A_c'][:]

#     # W = f['W'][:]
#     # QCQI = f['QCQI'][:]
#     Z = f['Z'][:]
#     Y = f['Y'][:]
#     X = f['X'][:]

# # #Making Time Matrix
# # rows, cols = A.shape[0], A.shape[1]
# # T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [17]:
# #TESTING CHECKING THAT THRESHOLDS WORKED
# w_data=data['w'].interp(zf=data['zh']).data #interpolation w data z coordinate from zh to zf
# variable='qc'; qc_data=data[variable].data # get qc data
# variable='qi'; qi_data=data[variable].data # get qc data
# qc_plus_qi=qc_data+qi_data

# def test_thresholds(t,type):
#     # w_data=data['w'].interp(zf=data['zh']).data

#     if type=='g':
#         A=A_g
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'max qcqi is {hey.max()}')
#     if type=='c':
#         A=A_c
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min qcqi is {hey.min()}')

# #GENERAL UPDRAFT
# t=35
# test_thresholds(t,'g')
# print('\n')
# #CLOUDY UPDRAFT
# test_thresholds(t,'c')